# LOL Draft Embedding Project — 전처리 파이프라인 스케치

In [138]:
"""
League Of Legend (LOL) 패치 25.19 기준 데이터 수집
Pipeline 1 : DDragon 15.19.1에서 25.19 패치 기준 챔피언 정보 및 스탯을 가져와 champion feature matrix 구축
Pipeline 2 : Oracle's Elixir 2025 (pro)에서 대회 픽 데이터를 수집하여 draft tensor 구축
Pipeline 3 : nathansmallcalder의 solo rank 픽 데이터(각 티어별)를 수집하여 draft tensor + tier filter 구축
Pipeline 4 : 공통 LabelEncoder & 검증
 
실행 환경: Python 3.10+, pandas, numpy, requests, scikit-learn, torch
"""

import csv
import io
import json
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [139]:
OE_PATH         = "2025_LoL_esports_match_data_from_OraclesElixir.csv"
SR_PATH         = "League of Legends(LoL) Matches Patch 25.19+"
MATCH_TBL       = f"{SR_PATH}/MatchTbl.csv"
TEAM_MATCH_TBL  = f"{SR_PATH}/TeamMatchTbl.csv"
CHAMPION_TBL    = f"{SR_PATH}/ChampionTbl.csv"
RANK_TBL        = f"{SR_PATH}/RankTbl.csv"

In [140]:
OUT_PRO         = "df_pro_enc.csv"
OUT_SOLO_ALL    = "df_solo_all_enc.csv"
OUT_SOLO_HIGH   = "df_solo_high_enc.csv"
OUT_EMBED       = "embedding_init.pt"
OUT_LE          = "label_encoder.pkl"
LOG_PATH        = "preprocessing_log.txt"

In [141]:
# DDragon
DDRAGON_VERSION = "15.19.1"

# DDragon tag 목록 (순서 고정)
ALL_TAGS = ["Fighter", "Tank", "Mage", "Assassin", "Marksman", "Support"]

# 수치 스탯 컬럼 (정규화 대상)
STAT_COLS = ["hp", "armor", "spellblock", "attackdamage",
                   "attackspeed", "movespeed", "hpregen"]

EMBED_DIM = 32

In [142]:
# 로그 유틸리티
_log_buffer = io.StringIO()
 
def log(msg: str = ""):
    """txt 버퍼에 한 줄 기록 (stdout 출력 없음)"""
    _log_buffer.write(msg + "\n")
 
def flush_log(path: str = LOG_PATH):
    with open(path, "w", encoding="utf-8") as f:
        f.write(_log_buffer.getvalue())

# Pipeline 1: DDragon → Champion Feature Matrix

In [143]:
def fetch_ddragon_features() -> pd.DataFrame:
    """
    DDragon 15.19.1 JSON → 챔피언별 특성 DataFrame 반환
    
    컬럼 구성 (총 17차원):
      - tag_Fighter, tag_Tank, tag_Mage, tag_Assassin,
        tag_Marksman, tag_Support           (one-hot, 6)
      - info_attack, info_defense, info_magic, info_difficulty
                                            (0~10 정규화, 4)
      - hp, armor, spellblock, attackdamage,
        attackspeed, movespeed, hpregen     (MinMaxScaler, 7)
    
    index: champion_name (소문자, 공백 제거)

    Returns
    -------
    pd.DataFrame, index=champion_name, columns=(tag × 6, info × 4, stats × 7)
    shape: (~170, 17), 모든 값 [0, 1] 정규화 완료
    """
    
    log("=" * 60)
    log("STEP 1: DDragon Feature Matrix")

    # 1. 파일 경로 설정 (파일명이 'champion.json'이라고 가정)
    file_path = 'champion.json'

    # 2. 로컬 파일 열기 및 로드
    with open(file_path, 'r', encoding='utf-8') as f:
        data_from_file = json.load(f)

    # 3. 기존처럼 "data" 키의 값 추출
    raw = data_from_file["data"]  # {"Aatrox": {...}, "Ahri": {...}, ...}
 
    records = []
    for name, champ in raw.items():
        # --- tag one-hot ---
        tag_vec  = {f"tag_{t}": int(t in champ["tags"]) for t in ALL_TAGS}
 
        # --- info 수치 (DDragon: 0~10 정수) ---
        info = champ["info"]
        info_vec = {
            "info_attack":     info["attack"]     / 10.0,
            "info_defense":    info["defense"]    / 10.0,
            "info_magic":      info["magic"]      / 10.0,
            "info_difficulty": info["difficulty"] / 10.0,
        }
 
        # --- base stats ---
        stats = champ["stats"]
        stat_vec = {col: stats.get(col, 0.0) for col in STAT_COLS}
 
        records.append({
            "champion_key":  champ["key"],          # Riot 내부 숫자 ID (문자열)
            "champion_name": name,                   # e.g. "Jinx", "MonkeyKing"
            **tag_vec, **info_vec, **stat_vec
        })
 
    df = pd.DataFrame(records).set_index("champion_name")

    # champion_key는 Riot 내부 숫자 ID로 feature가 아님 — 명시적 제거
    if "champion_key" in df.columns:
        df = df.drop(columns=["champion_key"])
 
    # MinMaxScaler: stat_cols만 적용
    scaler = MinMaxScaler()
    df[STAT_COLS] = scaler.fit_transform(df[STAT_COLS])

    log(f"  챔피언 수  : {len(df)}")
    log(f"  feature 차원: {df.shape[1]}  "
        f"(tag×6, info×4, stats×7)")
    log(f"  컬럼 목록  : {df.columns.tolist()}")
    log()
 
    return df  # shape: (~170, 17)

# Pipeline 2: Oracle's Elixir 2025 (프로 대회)

In [144]:
def load_oracles_elixir(path: str) -> pd.DataFrame:
    """
    Oracle's Elixir 2025 CSV → 모델 입력 형태로 변환
    
    핵심 처리:
      1. team row만 필터 (position == 'team')
      2. Blue/Red side별로 pick1~5 + result 추출
      3. gameid 기준으로 Blue/Red를 한 행으로 병합
    
    Returns:
      DataFrame with columns:
        blue_p1~p5, red_p1~p5  (champion name 문자열)
        result                  (1=blue win, 0=red win)
        league, patch           (메타데이터)
    """

    log("=" * 60)
    log("STEP 2: Oracle's Elixir 2025")
    log(f"  파일: {path}")

    df = pd.read_csv(path, low_memory=False)
 
    # team row만 (position이 'team'인 행)
    team = df[df["position"] == "team"].copy()

    log(f"  전체 행 수       : {len(df):,}")
    log(f"  team row 수      : {len(team):,}  (예상: 전체의 ~1/6)")
 
    # 필요 컬럼 선택
    pick_cols = [f"pick{i}" for i in range(1, 6)]
    keep      = ["gameid", "side", "result",
                 *pick_cols, "league", "patch"]
    team = team[keep].dropna(subset=pick_cols)
 
    # 챔피언 이름 정규화: 소문자·공백 제거 (DDragon key와 맞추기 위함)
    for col in pick_cols:
        team[col] = team[col].str.strip()
 
    # Blue / Red 분리 후 gameid 기준으로 merge
    blue = (team[team["side"] == "Blue"]
            .rename(columns={f"pick{i}": f"blue_p{i}" for i in range(1,6)})
            [["gameid","league","patch",
              "blue_p1","blue_p2","blue_p3","blue_p4","blue_p5","result"]])
 
    red  = (team[team["side"] == "Red"]
            .rename(columns={f"pick{i}": f"red_p{i}" for i in range(1,6)})
            [["gameid",
              "red_p1","red_p2","red_p3","red_p4","red_p5"]])
    
    merged = blue.merge(red, on="gameid", how="inner")
 
     # result 검증: 1=blue win, 0=red win, 균형 확인
    vc = merged["result"].value_counts().sort_index()
    log(f"  병합 게임 수     : {len(merged):,}")
    log(f"  result 분포      :\n"
        + "\n".join(f"    {k}: {v:,}" for k, v in vc.items()))
    log(f"  리그 분포 (상위 10):")
    for league, cnt in merged["league"].value_counts().head(10).items():
        log(f"    {league:<20}: {cnt:,}")
    log()
    return merged

# Pipeline 3: nathansmallcalder (솔로랭크)
### League of Legends(LoL) Matches Patch 25.19+ (https://www.kaggle.com/datasets/nathansmallcalder/lol-match-history-and-summoner-data-80k-matches?select=MatchTbl.csv)

In [145]:
def load_rank_table(rank_tbl_path: str) -> dict:
    """
    RankTbl.csv (RankId, RankName) → {int(RankId): RankName} 딕셔너리 반환
    """
    rank_names = {}
    with open(rank_tbl_path, mode='r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rank_names[int(row['RankId'])] = row['RankName']
    return rank_names

def load_solo_rank(match_tbl_path: str,
                   team_match_tbl_path: str,
                   champion_tbl_path: str,
                   rank_tbl_path: str,
                   rank_name_filter: list = None,
                   label: str = "전체") -> pd.DataFrame:
    """
    nathansmallcalder 3개 테이블 → 모델 입력 형태로 변환
    
    Args:
        match_tbl_path      : MatchTbl.csv
        team_match_tbl_path : TeamMatchTbl.csv
        champion_tbl_path   : ChampionTbl.csv
        rank_tbl_path       : RankTbl.csv
        rank_name_filter    : None이면 전체, [8,9,10] 이면 GM+Challenger만
                              RankFk 코드: 0=Unranked, 1=Iron ... 9=GM, 10=Challenger
        label               : 티어 지정 
    
    Returns:
      DataFrame with columns:
        blue_p1~p5, red_p1~p5 (champion name 문자열)
        result                 (1=blue win, 0=red win)
        rank_fk                (원본 티어 코드, 분석용)
    """
    log("=" * 60)
    log(f"STEP 3: Solo Rank  [{label}]")

    # 1. 테이블 로드
    match   = pd.read_csv(match_tbl_path)      # MatchId, Patch, QueueType, RankFk, GameDuration
    tms = pd.read_csv(team_match_tbl_path)
    champ   = pd.read_csv(champion_tbl_path)   # ChampionId, ChampionName
    rank_map = load_rank_table(rank_tbl_path)  # {int: str}

 
    # champion ID → name 딕셔너리
    id2name = dict(zip(champ["ChampionId"], champ["ChampionName"]))

    log(f"  MatchTbl 행 수         : {len(match):,}")
    log(f"  TeamMatchTbl 행 수     : {len(tms):,}")
    log(f"  ChampionTbl 챔피언 수  : {len(champ)}")
    log(f"  RankTbl 티어 목록      : " + ", ".join(f"{k}={v}" for k, v in sorted(rank_map.items())))

    # 2. QueueType 필터: CLASSIC 고정
    before_q = len(match)
    match = match[match["QueueType"] == "CLASSIC"]
    log(f"  QueueType 필터 (CLASSIC): {before_q:,} → {len(match):,}")
 
    # 3. RankFk → RankName 컬럼 부착 (map 방식)
    match = match.copy()
    match["RankName"] = match["RankFk"].map(rank_map)

    # 4. RankName 필터
    if rank_name_filter is not None:
        before_r = len(match)
        match = match[match["RankName"].isin(rank_name_filter)]
        log(f"  RankName 필터          : {rank_name_filter}  "
            f"({before_r:,} → {len(match):,})")
 
    valid_ids = set(match["MatchId"])
    tms = tms[tms["MatchFk"].isin(valid_ids)].copy()
    log(f"  필터 후 TeamMatch 행 수: {len(tms):,}")
 
    # 5. Champion ID → Name 변환
    # TeamMatchStatsTbl 컬럼: MatchID, B1Champ~B5Champ, R1Champ~R5Champ, Win
    # Win: 1=Blue win, 0=Red win
    b_cols = [f"B{i}Champ" for i in range(1, 6)]
    r_cols = [f"R{i}Champ" for i in range(1, 6)]
 
    for col in b_cols + r_cols:
        tms[col] = tms[col].map(id2name)
 
    before = len(tms)
    tms = tms.dropna(subset=b_cols + r_cols)
    dropped = before - len(tms)
    if dropped:
        log(f"  챔피언 ID 미매칭 제거  : {dropped}건")
     
    # 6. BlueWin/RedWin → result (1=Blue win, 0=Red win)
    # BlueWin=1이면 result=1, RedWin=1이면 result=0
    tms["result"] = tms["BlueWin"].astype(int)
    tms = tms.rename(columns={
        **{f"B{i}Champ": f"blue_p{i}" for i in range(1, 6)},
        **{f"R{i}Champ": f"red_p{i}"  for i in range(1, 6)},
    })
 
    # 7.MatchTbl 메타 컬럼 병합 (RankFk, RankName, Patch)
    tms = tms.merge(
        match[["MatchId", "RankFk", "RankName", "Patch"]],
        left_on="MatchFk", right_on="MatchId",
        how="left"
    ).rename(columns={"RankFk": "rank_fk", "RankName": "rank_name"})
 
    pick_cols = [f"blue_p{i}" for i in range(1, 6)] + \
                [f"red_p{i}"  for i in range(1, 6)]
    tms = tms[["MatchFk", *pick_cols, "result", "rank_fk", "rank_name", "Patch"]]
 
    log(f"  최종 게임 수           : {len(tms):,}")
    log(f"  result 분포            :")
    for v, c in tms["result"].value_counts().sort_index().items():
        log(f"    {v}: {c:,}")
    log(f"  티어 분포 (RankName)   :")
    for name, cnt in tms["rank_name"].value_counts().items():
        log(f"    {name:<16}: {cnt:,}")
    log()
    
    return tms

# Pipeline 4: 공통 LabelEncoder & ID 변환

In [146]:
# 매치 데이터 표시명 → DDragon 내부 key 매핑
# (공백·특수문자 차이로 인한 미매칭 21개)
DDRAGON_NAME_MAP = {
    "Aurelion Sol":   "AurelionSol",
    "Bel'Veth":       "Belveth",
    "Cho'Gath":       "Chogath",
    "Dr. Mundo":      "DrMundo",
    "Jarvan IV":      "JarvanIV",
    "K'Sante":        "KSante",
    "Kai'Sa":         "Kaisa",
    "Kha'Zix":        "Khazix",
    "Kog'Maw":        "KogMaw",
    "LeBlanc":        "Leblanc",
    "Lee Sin":        "LeeSin",
    "Master Yi":      "MasterYi",
    "Miss Fortune":   "MissFortune",
    "Nunu & Willump": "Nunu",
    "Rek'Sai":        "RekSai",
    "Renata Glasc":   "Renata",
    "Tahm Kench":     "TahmKench",
    "Twisted Fate":   "TwistedFate",
    "Vel'Koz":        "Velkoz",
    "Wukong":         "MonkeyKing",
    "Xin Zhao":       "XinZhao",
}
 
# DDragon 15.19.1 이후 추가된 미지원 챔피언 → 전처리 단계에서 제거
UNKNOWN_CHAMPIONS = {"Zaahen"}

In [147]:
def build_champion_encoder(*dfs: pd.DataFrame) -> LabelEncoder:
    """
    여러 DataFrame에 등장하는 모든 챔피언명을 통합하여 LabelEncoder 생성
 
    NOTE: 두 데이터셋을 동일 encoder로 처리해야
          임베딩 인덱스 일관성이 유지되어 공간 비교가 성립함.
 
    Returns: LabelEncoder  (le.classes_ = 정렬된 챔피언명 배열)
    """
    log("=" * 60)
    log("STEP 4: Champion LabelEncoder 구성")
 
    pick_cols = [f"blue_p{i}" for i in range(1,6)] + \
                [f"red_p{i}"  for i in range(1,6)]
 
    all_names = pd.concat([
        pd.Series(d[pick_cols].values.flatten()) for d in dfs
    ]).dropna()

    # DDragon 미지원 챔피언(패치 이후 추가) 제거
    unique_before  = set(all_names.unique())
    removed_names  = unique_before & UNKNOWN_CHAMPIONS        # 실제 제거 이름 집합
    all_names      = sorted(unique_before - UNKNOWN_CHAMPIONS)
    if removed_names:
        log(f"  미지원 챔피언 제거: {removed_names}  ({len(removed_names)}개)")
 
    le = LabelEncoder()
    le.fit(all_names)
 
    log(f"  통합 챔피언 풀 크기: {len(le.classes_)}")
    log()

    return le

In [148]:
def encode_draft_df(df: pd.DataFrame,
                    le: LabelEncoder,
                    label: str = "") -> pd.DataFrame:
    """
    champion name (str) → champion ID (int) 변환
    미등록 챔피언 포함 행은 제거 후 로그 기록
 
    Returns: 인코딩된 DataFrame (pick 컬럼이 int로 교체됨)
    """
    pick_cols = [f"blue_p{i}" for i in range(1,6)] + \
                [f"red_p{i}"  for i in range(1,6)]
    known  = set(le.classes_)
    before = len(df)
 
    # 알 수 없는 챔피언이 포함된 행 제거
    mask = df[pick_cols].apply(lambda col: col.isin(known)).all(axis=1)
    df   = df[mask].copy()
 
    for col in pick_cols:
        df[col] = le.transform(df[col])
 
    removed = before - len(df)
    log(f"  [{label}] 인코딩 후 행 수: {len(df):,}  "
        f"(제거: {removed}건)")
    return df

# Pipeline 5: Embedding 초기화 Weight 생성

In [149]:
def build_embedding_init(feat_df: pd.DataFrame,
                         le: LabelEncoder,
                         embed_dim: int = EMBED_DIM) -> torch.Tensor:
    """
    DDragon feature matrix → Embedding 초기 weight Tensor
 
    처리 순서:
      1. LabelEncoder 순서에 맞게 feature matrix 정렬
      2. Linear(17 → embed_dim) 투영 (Xavier 초기화)
      3. 결과 Tensor 반환
 
    Returns
    -------
    torch.Tensor, shape: (NUM_CHAMPIONS, embed_dim)
    사용법:
      self.embed = nn.Embedding(NUM_CHAMPIONS, embed_dim)
      self.embed.weight = nn.Parameter(init_weight.clone())
    """
    log("=" * 60)
    log("STEP 5: DDragon Embedding Init Weight 생성")
 
    num_champs = len(le.classes_)
    feat_dim   = feat_df.shape[1]  # 17
 
    ordered = np.zeros((num_champs, feat_dim), dtype=np.float32)
    missing = []
    for idx, name in enumerate(le.classes_):
        # 1순위: 표시명 직접 매칭
        # 2순위: DDRAGON_NAME_MAP을 통한 DDragon key 매칭
        ddragon_key = DDRAGON_NAME_MAP.get(name, name)
        if ddragon_key in feat_df.index:
            ordered[idx] = feat_df.loc[ddragon_key].values
        elif name in feat_df.index:
            ordered[idx] = feat_df.loc[name].values
        else:
            missing.append(name)
 
    nonzero = int((ordered.sum(axis=1) != 0).sum())
    if missing:
        log(f"  DDragon 미등록 챔피언 (0벡터 처리): {missing}")
    log(f"  DDragon 매칭 성공: {nonzero} / {num_champs}  "
        f"(0벡터: {num_champs - nonzero})")
 
    feat_tensor = torch.tensor(ordered)            # (N, 17)
 
    proj = nn.Linear(feat_dim, embed_dim, bias=False)
    nn.init.xavier_uniform_(proj.weight)
    with torch.no_grad():
        weight = proj(feat_tensor)                 # (N, embed_dim)
 
    log(f"  챔피언 수   : {num_champs}")
    log(f"  feature 차원: {feat_dim}  →  embed_dim: {embed_dim}")
    log(f"  weight shape: {tuple(weight.shape)}")
    log(f"  저장 파일   : {OUT_EMBED}")
    log()
    
    return weight

# 실행 진입점

In [150]:
log("LOL Draft Embedding — Preprocessing Pipeline v2")
log(f"DDragon Version: {DDRAGON_VERSION}")
log()

# Step 1: DDragon
feat_df = fetch_ddragon_features()

# Step 2: 프로 (OE 2025)
df_pro = load_oracles_elixir(OE_PATH)

# Step 3: 솔로랭크
df_solo_all = load_solo_rank(
    MATCH_TBL, TEAM_MATCH_TBL, CHAMPION_TBL, RANK_TBL,
    rank_name_filter=None,
    label="전체"
)
df_solo_high = load_solo_rank(
    MATCH_TBL, TEAM_MATCH_TBL, CHAMPION_TBL, RANK_TBL,
    rank_name_filter=["Grandmaster", "Challenger"],
    label="GM+Challenger"
)

# Step 4: 통합 Encoder
le = build_champion_encoder(df_pro, df_solo_all)

log("=" * 60)
log("STEP 4 (encode): 각 DataFrame 인코딩")
df_pro_enc       = encode_draft_df(df_pro,       le, "Pro OE2025")
df_solo_all_enc  = encode_draft_df(df_solo_all,  le, "Solo All")
df_solo_high_enc = encode_draft_df(df_solo_high, le, "Solo GM+Chall")
log()

# Step 5: Embedding init weight
init_weight = build_embedding_init(feat_df, le, embed_dim=EMBED_DIM)

# ── Export ──────────────────────────────────────
log("=" * 60)
log("EXPORT 결과물")

df_pro_enc.to_csv(OUT_PRO,        index=False)
log(f"  {OUT_PRO:<30}: {df_pro_enc.shape}  columns={df_pro_enc.columns.tolist()}")

df_solo_all_enc.to_csv(OUT_SOLO_ALL,  index=False)
log(f"  {OUT_SOLO_ALL:<30}: {df_solo_all_enc.shape}")

df_solo_high_enc.to_csv(OUT_SOLO_HIGH, index=False)
log(f"  {OUT_SOLO_HIGH:<30}: {df_solo_high_enc.shape}")

torch.save(init_weight, OUT_EMBED)
log(f"  {OUT_EMBED:<30}: shape={tuple(init_weight.shape)}")

with open(OUT_LE, "wb") as f:
    pickle.dump(le, f)
log(f"  {OUT_LE:<30}: {len(le.classes_)} 챔피언")

log()
log("=" * 60)
log("최종 요약")
log(f"  프로 게임 수         : {len(df_pro_enc):,}")
log(f"  솔로랭크 전체        : {len(df_solo_all_enc):,}")
log(f"  솔로랭크 GM+Chall    : {len(df_solo_high_enc):,}")
log(f"  챔피언 풀            : {len(le.classes_)}")
log(f"  임베딩 차원          : {EMBED_DIM}")
log()
log("다음 단계: DraftEmbeddingFFNN 모델 정의 및 학습 루프")

# 로그 파일 저장
flush_log(LOG_PATH)
# 실행 완료 메시지만 콘솔 출력
print(f"완료. 로그: {LOG_PATH}")

완료. 로그: preprocessing_log.txt
